# Starting Kit

<div style="background-color:#eef6ff;color:#000;padding:20px;border-radius:10px;border-left:6px solid #2b7de9">

## Challenge Overview

This competition focuses on **predicting the self-reported happiness level of individuals** using survey responses from the **World Values Survey (WVS)**.

The dataset comes from the **World Values Survey Time-Series Dataset (1981–2022), Version V3.0**, which aggregates survey waves conducted worldwide over four decades.

Each row corresponds to **one individual respondent** from a nationally representative survey sample.

The goal is to predict the target variable:

**A008 — Self-reported Happiness Score**

using demographic, social, cultural, and economic indicators collected in the survey.

This challenge highlights the intersection between:

- Social science research
- Tabular machine learning
- Feature engineering on high-dimensional survey data

</div>

<div style="background-color:#eef6ff;color:#000;padding:20px;border-radius:10px;border-left:6px solid #2b7de9">
## Dataset Splitting Strategy

The dataset is divided into three files:

| File | Purpose |
|-----|-----|
| `train.csv` | Training data with labels |
| `test_public.csv` | Used for public leaderboard evaluation |
| `test_private.csv` | Used for final ranking |

The corresponding labels are available for:

- `train.csv`
- `test_public_labels.csv`

The labels for **test_private.csv** remain hidden and are used only for final evaluation.

### Splitting Method

The split was performed using **stratification on the happiness score**.

This ensures:

- Balanced representation of happiness levels
- Stable evaluation across classes
- Fair comparison between models

Preserving the ordinal distribution of the target variable is essential for reliable benchmarking.

</div>

<div style="background-color:#eef6ff;color:#000;padding:20px;border-radius:10px;border-left:6px solid #2b7de9">
## Evaluation Metrics

Because happiness is an **ordinal variable**, the evaluation metrics must capture both **agreement** and **prediction distance**.

Two complementary metrics are used.

### Quadratic Weighted Kappa (QWK)

Quadratic Weighted Kappa measures the **agreement between predicted and true labels**, while penalizing larger ordinal mistakes more heavily.

For example:

Predicting *Very Happy* instead of *Happy* is less severe than predicting *Very Unhappy*.

QWK ranges from:

- **1** → Perfect agreement  
- **0** → Random agreement  
- **-1** → Complete disagreement

This is the **primary ranking metric** used for the leaderboard.

### Mean Absolute Error (MAE)

MAE measures the **average absolute difference** between predicted and true happiness scores.

This metric provides:

- A simple and interpretable measure of prediction error
- Sensitivity to the magnitude of mistakes

### Why Use Both?

Using both metrics ensures models:

- Respect the **ordinal structure** of the target (QWK)
- Produce **numerically accurate predictions** (MAE)

The **public leaderboard** is computed on `test_public`.

The **final ranking** is computed on `test_private`.

</div>

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
wvs_path = "dev_phase/input_data/train.csv"
df = pd.read_csv(wvs_path)
df.head()

,version,doi,S001,S002VS,S003,COUNTRY_ALPHA,COW_NUM,COW_ALPHA,S004,S006,...,X051,X052,X053,X054,X055,X048ISO,X003R,X003R2,X050C,A008
0,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,7,364,IRN,630,IRN,NaN,364071459,...,364001.0,1.0,NaN,NaN,NaN,364007.0,6.0,3.0,1.0,2
1,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,2,724,ESP,230,SPN,NaN,1249,...,NaN,NaN,NaN,NaN,NaN,724019.0,1.0,1.0,NaN,2
2,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,4,400,JOR,663,JOR,NaN,476,...,400002.0,NaN,NaN,NaN,NaN,400003.0,2.0,2.0,NaN,2
3,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,4,682,SAU,670,SAU,NaN,343,...,682002.0,NaN,NaN,NaN,NaN,682002.0,4.0,3.0,NaN,2
4,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,6,356,IND,750,IND,NaN,1456,...,356019.0,2.0,6.0,7.0,8.0,356014.0,4.0,3.0,NaN,3


In [12]:
df["A008"].unique()

array([2, 3, 1, 4], dtype=int64)

In [13]:
import gc

df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)
gc.collect()
df.head()

,version,doi,S001,S002VS,S003,COUNTRY_ALPHA,COW_NUM,COW_ALPHA,S004,S006,...,X051,X052,X053,X054,X055,X048ISO,X003R,X003R2,X050C,A008
0,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,5,364,IRN,630,IRN,NaN,670,...,364001.0,2.0,10.0,6.0,9.0,364007.0,4.0,2.0,NaN,2
1,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,4,566,NGA,475,NIG,NaN,1031,...,566072.0,NaN,NaN,NaN,NaN,NaN,3.0,2.0,NaN,1
2,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,6,275,PSE,667,PSE,NaN,756,...,NaN,1.0,9.0,1.0,3.0,275008.0,3.0,2.0,NaN,2
3,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,5,840,USA,2,USA,NaN,114,...,840001.0,-3.0,-3.0,-3.0,-3.0,NaN,6.0,3.0,NaN,2
4,5-0-0 (2024-04-30),doi.org/10.14281/18241.25,2,5,818,EGY,651,EGY,NaN,1550,...,NaN,-3.0,-3.0,-3.0,-3.0,818018.0,6.0,3.0,2.0,2


### Simple Preprocessing + Baseline model

In [14]:
import pandas as pd
import numpy as np

# Replace missing codes
missing_codes = [-1, -2, -4, -5]
df = df.replace(missing_codes, np.nan)

# Keep numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Target
y = numeric_df["A008"]

# Compute correlation column-wise (much faster)
correlations = numeric_df.apply(
    lambda col: col.corr(y, method="spearman")
)

# Remove self-correlation and sort
top_corr = (
    correlations
    .drop("A008")
    .dropna()
    .abs()
    .sort_values(ascending=False)
)

print(top_corr.head(10))

c:\Users\PC\anaconda3\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


F187       0.600099
E213       0.593834
A124_31    0.569275
G007_56    0.468994
G007_51    0.446078
G007_07    0.436894
G007_06    0.431764
G007_16    0.424241
G007_11    0.419552
D002       0.403205
dtype: float64


In [15]:
features = top_corr.head(30).index.tolist()

In [16]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import cohen_kappa_score



X = df[features]
y = df["A008"]
mask = y.notna()
X = X[mask]
y = y[mask]

# Train/validation split (stratified)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Impute missing feature values
imputer = SimpleImputer(strategy="most_frequent")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=features
)

X_val = pd.DataFrame(
    imputer.transform(X_val),
    columns=features
)

# Strong tree model
model = HistGradientBoostingClassifier(
                max_depth=8,
                max_iter=100,
                random_state=42
            )

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_val)

# Macro F1
scores = {
    "qwk": float(cohen_kappa_score(y_val, y_pred, weights="quadratic")),
    "mae": float(mean_absolute_error(y_val, y_pred)),
}
print(f"QWK: {scores['qwk']:.4f}  | MAE: {scores['mae']:.4f}")

QWK: 0.3597  | MAE: 0.4555
